In [1]:
import numpy as np
import sklearn
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import imageio.v3 as iio
import os
from PIL import Image
import imagehash


# Data loading and Processing

In [ ]:
Batch = 32
Random_seed = 42

train_Bi_raw = tf.keras.utils.image_dataset_from_directory(
    "Training_Bi/",
    validation_split=0.15,
    subset="training",
    seed=Random_seed,
    batch_size=None
)

val_Bi_raw = tf.keras.utils.image_dataset_from_directory(
    "Training_Bi/",
    validation_split=0.15,
    subset="validation",
        seed=Random_seed,
        batch_size=None
)

test_Bi_raw = tf.keras.utils.image_dataset_from_directory(
    "Test/",
    batch_size=None,
    shuffle=False
)



Found 39375 files belonging to 2 classes.
Using 33469 files for training.
Found 39375 files belonging to 2 classes.
Using 5906 files for validation.
Found 8617 files belonging to 2 classes.


In [3]:
def focal_loss(gamma, alpha):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_weight = alpha * tf.pow(1 - p_t, gamma)
        return tf.reduce_mean(focal_weight * bce)
    return loss


def augment(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.rot90(image, k=tf.random.uniform((), 0, 4, dtype=tf.int32))
    image = tf.image.random_brightness(image, max_delta=0.3)
    image = tf.image.random_contrast(image, lower=0.7, upper=1.4)
    image = tf.image.random_saturation(image, lower=0.6, upper=1.6)
    image = tf.image.random_hue(image, max_delta=0.05)
    
    # Haze - fixed with tf.cond
    haze_intensity = tf.random.uniform((), 0.05, 0.25)
    haze = tf.ones_like(image) * 0.7
    image = tf.cond(
        tf.random.uniform(()) > 0.5,
        lambda: image * (1 - haze_intensity) + haze * haze_intensity,
        lambda: image
    )

    # Blur - fixed with tf.cond
    def apply_blur(img):
        img = tf.expand_dims(img, 0)
        img = tf.nn.avg_pool2d(img, ksize=3, strides=1, padding='SAME')
        return tf.squeeze(img, 0)

    image = tf.cond(
        tf.random.uniform(()) > 0.6,
        lambda: apply_blur(image),
        lambda: image
    )

    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def compute_class_weights(ds):
    labels = np.array([y.numpy() for _, y in ds])
    counts = np.bincount(labels)
    total = len(labels)
    weights = {i: total / (len(counts) * c) for i, c in enumerate(counts)}
    print(f"Class weights: {weights}")
    return weights

In [4]:
class_weights = compute_class_weights(train_Bi_raw)

train_ds = (
    train_Bi_raw
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(Batch)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    val_Bi_raw
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(Batch)
    .prefetch(tf.data.AUTOTUNE)
)
test_ds = (
    test_Bi_raw
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(Batch)
    .prefetch(tf.data.AUTOTUNE)
)

Class weights: {0: 0.787542943197327, 1: 1.3694353518821605}


In [9]:
model = keras.models.Sequential([
    
    keras.layers.Conv2D(8, 3, activation='relu', padding='same',strides=2),
    keras.layers.Conv2D(8, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(8, 3, activation='relu', padding='same',strides=2),
    keras.layers.BatchNormalization(),
    keras.layers.Flatten(),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=[
        'accuracy',
        keras.metrics.Recall(name='recall'),
        keras.metrics.Precision(name='precision'),
    ]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7
    ),
    keras.callbacks.ModelCheckpoint(
        'best_fire_model.keras',
        monitor='val_recall',
        save_best_only=True,
    )
]

history = model.fit(
    train_ds,
    epochs=30,
    validation_data=val_ds,
    class_weight=None,
    callbacks=callbacks
)

Epoch 1/30
1046/1046 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.9027 - loss: 0.0187 - precision: 0.8894 - recall: 0.8376 - val_accuracy: 0.9455 - val_loss: 0.0117 - val_precision: 0.9830 - val_recall: 0.8643 - learning_rate: 1.0000e-04
Epoch 2/30
1046/1046 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.9422 - loss: 0.0109 - precision: 0.9608 - recall: 0.8777 - val_accuracy: 0.9543 - val_loss: 0.0091 - val_precision: 0.9942 - val_recall: 0.8788 - learning_rate: 1.0000e-04
Epoch 3/30
1046/1046 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.9517 - loss: 0.0090 - precision: 0.9640 - recall: 0.9014 - val_accuracy: 0.9653 - val_loss: 0.0073 - val_precision: 0.9974 - val_recall: 0.9064 - learning_rate: 1.0000e-04
Epoch 4/30
1046/1046 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9615 - loss: 0.0076 - precision: 0.9706 - recall: 0.9226 - val_accuracy: 0.9675 - val_loss: 0.0061 - val_precision: 0.9914 - val_recall: 0.9181 - learning_rate: 1.0000e-04
Epoch 5/30
1046/1046 ━━━━━━━━━━━━━━━━━━━

In [10]:
loss, acc, recall, precision = model.evaluate(test_ds)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test Precision: {precision:.4f}")

# Get probabilities
y_pred_prob = model.predict(test_ds)

# 2. Confusion matrix
true_labels = np.concatenate([y.numpy() for _, y in test_ds])

pred_classes = (y_pred_prob > 0.35).astype(int).flatten()
cm = confusion_matrix(true_labels, pred_classes)
print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(true_labels, pred_classes, target_names=['No_Fire', 'Fire']))

270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6259 - loss: 0.0829 - precision: 0.5766 - recall: 0.2770   
Test Loss: 0.0829
Test Accuracy: 0.6259
Test Recall: 0.2770
Test Precision: 0.5766
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Confusion Matrix:
[[3557 1580]
 [1841 1639]]

Classification Report:
              precision    recall  f1-score   support

     No_Fire       0.66      0.69      0.68      5137
        Fire       0.51      0.47      0.49      3480

    accuracy                           0.60      8617
   macro avg       0.58      0.58      0.58      8617
weighted avg       0.60      0.60      0.60      8617



In [ ]:
for t in np.arange(0.25, 0.70, 0.05):
    pred_classes = (y_pred_prob > t).astype(int).flatten()
    print(f"Threshold {t:.2f}:", classification_report(true_labels, pred_classes, target_names=['No_Fire', 'Fire'], output_dict=True)['macro avg']['f1-score'])

Threshold 0.25: 0.6194930089123402
Threshold 0.30: 0.7366638952987804
Threshold 0.35: 0.7679141310456695
Threshold 0.40: 0.7662571221328429
Threshold 0.45: 0.7551334731433963
Threshold 0.50: 0.7264434452879647
Threshold 0.55: 0.696916519173784
Threshold 0.60: 0.6623694115826979
Threshold 0.65: 0.6140546932676885


270/270 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.7579 - loss: 0.0516 - precision: 0.8148 - recall: 0.5184
Test Loss: 0.0516
Test Accuracy: 0.7579
Test Recall: 0.5184
Test Precision: 0.8148
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Confusion Matrix:
[[4438  699]
 [1178 2302]]

Classification Report:
              precision    recall  f1-score   support

     No_Fire       0.79      0.86      0.83      5137
        Fire       0.77      0.66      0.71      3480

    accuracy                           0.78      8617
   macro avg       0.78      0.76      0.77      8617
weighted avg       0.78      0.78      0.78      8617